In [ ]:
# -*- coding: utf-8 -*-
"""RUDN NLP HW1 - Word2Vec + Character TF-IDF Ensemble
- Word2Vec (trained on the fly) + average pooling → Logistic Regression
- Character TF-IDF (ngram 3-6) → Logistic Regression
- Soft voting ensemble
No transformers, no MLP, no CountVectorizer, no 5-fold CV.
"""

import numpy as np
import pandas as pd
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import VotingClassifier
from sklearn.metrics import f1_score
from gensim.models import Word2Vec
from gensim.utils import simple_preprocess

# Download NLTK data
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

# ============================================================
# 1. Fixed seed for reproducibility
# ============================================================
SEED = 123
np.random.seed(SEED)

# ============================================================
# 2. Load data
# ============================================================
train_df = pd.read_csv("/kaggle/input/competitions/rudn-dl-nlp-hw-1-2026/train.csv")
test_df = pd.read_csv("/kaggle/input/competitions/rudn-dl-nlp-hw-1-2026/test.csv")
print(f"Train: {train_df.shape}, Test: {test_df.shape}")

# ============================================================
# 3. Preprocessing (different: lemmatization, keep some punctuation)
# ============================================================
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def preprocess_for_w2v(text):
    """Tokenize, remove stopwords, lemmatize, return list of tokens"""
    text = str(text).lower()
    # Remove [NAME] and URLs
    text = re.sub(r'\[.*?\]', '', text)
    text = re.sub(r'http\S+', '', text)
    # Keep letters and spaces (remove digits, keep apostrophes as they are)
    text = re.sub(r"[^a-z']", ' ', text)
    # Tokenize using gensim's simple_preprocess (handles punctuation)
    tokens = simple_preprocess(text, deacc=True)  # deacc=True removes accents
    # Remove stopwords and lemmatize
    tokens = [lemmatizer.lemmatize(tok) for tok in tokens if tok not in stop_words and len(tok) > 2]
    return tokens

def preprocess_for_tfidf(text):
    """Return string for TF-IDF (different cleaning)"""
    text = str(text).lower()
    text = re.sub(r'\[.*?\]', '', text)
    text = re.sub(r'http\S+', '', text)
    # Keep letters, apostrophes, and spaces (no digits)
    text = re.sub(r'[^a-z\']', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# Apply preprocessing
print("Preprocessing...")
train_tokens = train_df['text'].apply(preprocess_for_w2v)  # list of tokens
train_text_tfidf = train_df['text'].apply(preprocess_for_tfidf)
test_tokens = test_df['text'].apply(preprocess_for_w2v)
test_text_tfidf = test_df['text'].apply(preprocess_for_tfidf)

y = train_df['target']

# ============================================================
# 4. Train Word2Vec model on the training corpus
# ============================================================
print("Training Word2Vec...")
w2v_model = Word2Vec(
    sentences=train_tokens,
    vector_size=200,
    window=5,
    min_count=2,
    workers=4,
    sg=1,          # skip-gram
    seed=SEED
)
# Save model (optional)
# w2v_model.save("w2v.model")

def get_average_vector(tokens, model, vector_size=200):
    """Average word vectors for a document; return zero vector if no words"""
    vectors = [model.wv[word] for word in tokens if word in model.wv]
    if len(vectors) == 0:
        return np.zeros(vector_size)
    return np.mean(vectors, axis=0)

print("Creating average word vectors for train...")
X_train_w2v = np.array([get_average_vector(tok, w2v_model) for tok in train_tokens])
X_test_w2v = np.array([get_average_vector(tok, w2v_model) for tok in test_tokens])

# ============================================================
# 5. Character-level TF-IDF (ngram 3-6, char_wb)
# ============================================================
print("Creating character TF-IDF features...")
char_tfidf = TfidfVectorizer(
    analyzer='char_wb',
    ngram_range=(3, 6),
    max_features=15000,
    sublinear_tf=True
)
X_train_char = char_tfidf.fit_transform(train_text_tfidf)
X_test_char = char_tfidf.transform(test_text_tfidf)

# ============================================================
# 6. Train/validation split for evaluation (20%)
# ============================================================
# Split the averaged word2vec features and char features consistently
X_w2v_train, X_w2v_val, y_train, y_val = train_test_split(
    X_train_w2v, y, test_size=0.2, stratify=y, random_state=SEED
)
X_char_train, X_char_val, _, _ = train_test_split(
    X_train_char, y, test_size=0.2, stratify=y, random_state=SEED
)

# ============================================================
# 7. Train two separate Logistic Regression models
# ============================================================
lr_w2v = LogisticRegression(class_weight='balanced', max_iter=1000, C=0.5, random_state=SEED)
lr_w2v.fit(X_w2v_train, y_train)
pred_w2v_val = lr_w2v.predict(X_w2v_val)
f1_w2v = f1_score(y_val, pred_w2v_val, average='macro')
print(f"Word2Vec + LR validation macro F1: {f1_w2v:.4f}")

lr_char = LogisticRegression(class_weight='balanced', max_iter=1000, C=1.0, random_state=SEED)
lr_char.fit(X_char_train, y_train)
pred_char_val = lr_char.predict(X_char_val)
f1_char = f1_score(y_val, pred_char_val, average='macro')
print(f"Char TF-IDF + LR validation macro F1: {f1_char:.4f}")

# ============================================================
# 8. Soft voting ensemble
# ============================================================
# For validation, combine probabilities
prob_w2v = lr_w2v.predict_proba(X_w2v_val)
prob_char = lr_char.predict_proba(X_char_val)
avg_probs = (prob_w2v + prob_char) / 2
pred_ensemble = np.argmax(avg_probs, axis=1)
f1_ensemble = f1_score(y_val, pred_ensemble, average='macro')
print(f"Ensemble (soft vote) validation macro F1: {f1_ensemble:.4f}")

# ============================================================
# 9. Retrain on full training data
# ============================================================
print("Retraining on full training set...")
lr_w2v_full = LogisticRegression(class_weight='balanced', max_iter=1000, C=0.5, random_state=SEED)
lr_w2v_full.fit(X_train_w2v, y)

lr_char_full = LogisticRegression(class_weight='balanced', max_iter=1000, C=1.0, random_state=SEED)
lr_char_full.fit(X_train_char, y)

# ============================================================
# 10. Predict on test set
# ============================================================
prob_w2v_test = lr_w2v_full.predict_proba(X_test_w2v)
prob_char_test = lr_char_full.predict_proba(X_test_char)
avg_probs_test = (prob_w2v_test + prob_char_test) / 2
test_preds = np.argmax(avg_probs_test, axis=1)

# ============================================================
# 11. Create submission
# ============================================================
submission = pd.DataFrame({'id': test_df['id'], 'target': test_preds})
submission.to_csv('w2v_char_ensemble_submission.csv', index=False)
print("✅ Submission saved as 'w2v_char_ensemble_submission.csv'")